# Composite Factorization Patterns

How do the Collatz classifications of composite numbers relate to those of their prime factors?
We investigate semiprimes, prime powers, near-trivial composites, and whether a "class multiplication table" exists.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from collatz.core import orbit, total_stopping_time, stopping_time, stopping_destination
from collatz.dropping import (
    dropping_time, dropping_set, dropping_modulus, dropping_orbit,
    orbital_oddity, dropping_genus, classify_range as dropping_classify_range
)
from collatz.stopping import (
    stopping_class, stopping_point, stopping_signature,
    stopping_modulus_for_class, classify_range as stopping_classify_range
)
from collatz.factorization import (
    prime_factorization, is_prime, prime_class_character,
    compare_composite_vs_factors, factor_classification_table,
    shared_class_primes, class_distribution_by_prime
)
from collatz.geometry import (
    orbital_triple, complex_multiplier, proportional_power_ratio
)

plt.rcParams['figure.figsize'] = (12, 6)
print('All imports successful.')

## Semiprimes: n = p × q

In [ ]:
# For semiprimes up to 1000, compare class(pq) to class(p) and class(q).
# A semiprime is n = p * q where p and q are primes (p <= q).

primes_list = [p for p in range(2, 500) if is_prime(p)]

rows = []
for i, p in enumerate(primes_list):
    for q in primes_list[i:]:
        n = p * q
        if n > 1000:
            break
        if n <= 1:
            continue
        rows.append({
            'n': n,
            'p': p,
            'q': q,
            'class_n': dropping_set(n),
            'class_p': dropping_set(p),
            'class_q': dropping_set(q),
            'oddity_n': orbital_oddity(n),
            'oddity_p': orbital_oddity(p),
            'oddity_q': orbital_oddity(q),
        })

df_semi = pd.DataFrame(rows)
print(f"Found {len(df_semi)} semiprimes up to 1000.")

# Show whether composite class matches either factor class
df_semi['matches_p'] = df_semi['class_n'] == df_semi['class_p']
df_semi['matches_q'] = df_semi['class_n'] == df_semi['class_q']
df_semi['matches_any'] = df_semi['matches_p'] | df_semi['matches_q']

print(f"\nComposite class matches p's class: {df_semi['matches_p'].sum()} / {len(df_semi)} ({100*df_semi['matches_p'].mean():.1f}%)")
print(f"Composite class matches q's class: {df_semi['matches_q'].sum()} / {len(df_semi)} ({100*df_semi['matches_q'].mean():.1f}%)")
print(f"Composite class matches either factor: {df_semi['matches_any'].sum()} / {len(df_semi)} ({100*df_semi['matches_any'].mean():.1f}%)")

# Cross-tabulation: composite class vs factor class pairs
df_semi['factor_classes'] = list(zip(df_semi['class_p'], df_semi['class_q']))
ct = pd.crosstab(df_semi['factor_classes'].apply(str), df_semi['class_n'])
print("\nCross-tabulation of (class_p, class_q) vs class(n):")
print(ct.head(20))

df_semi.head(15)

## Prime Powers: n = p^k

In [ ]:
# For primes p < 100 and k = 2..5, compare class(p^k) to class(p).

primes_under_100 = [p for p in range(2, 100) if is_prime(p)]

rows_pp = []
for p in primes_under_100:
    class_p = dropping_set(p)
    oddity_p = orbital_oddity(p)
    for k in range(2, 6):
        pk = p ** k
        if pk > 100000:  # keep computation manageable
            continue
        rows_pp.append({
            'p': p,
            'k': k,
            'p_k': pk,
            'class_p': class_p,
            'class_pk': dropping_set(pk),
            'oddity_p': oddity_p,
            'oddity_pk': orbital_oddity(pk),
            'class_match': dropping_set(pk) == class_p,
        })

df_pp = pd.DataFrame(rows_pp)
print(f"Prime power entries: {len(df_pp)}")
print(f"\nClass(p^k) == Class(p) frequency: {df_pp['class_match'].sum()} / {len(df_pp)} ({100*df_pp['class_match'].mean():.1f}%)")

# Table: class(p) vs class(p^k) for each exponent k
for k in range(2, 6):
    subset = df_pp[df_pp['k'] == k]
    print(f"\n--- k = {k} ---")
    print(f"Class match rate: {subset['class_match'].mean():.2%}")
    print(subset[['p', 'p_k', 'class_p', 'class_pk', 'class_match']].head(10))

# Visualization: class(p^k) vs k for each prime
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter of class(p^k) vs class(p) colored by k
for k in range(2, 6):
    subset = df_pp[df_pp['k'] == k]
    axes[0].scatter(subset['class_p'], subset['class_pk'], alpha=0.6, label=f'k={k}', s=30)
axes[0].plot([0, df_pp['class_p'].max()], [0, df_pp['class_p'].max()], 'k--', alpha=0.3, label='y=x')
axes[0].set_xlabel('Dropping Set of p')
axes[0].set_ylabel('Dropping Set of p^k')
axes[0].set_title('Prime Power Classes: class(p^k) vs class(p)')
axes[0].legend()

# Right: oddity(p^k) vs oddity(p)
for k in range(2, 6):
    subset = df_pp[df_pp['k'] == k]
    axes[1].scatter(subset['oddity_p'], subset['oddity_pk'], alpha=0.6, label=f'k={k}', s=30)
axes[1].plot([0, df_pp['oddity_p'].max()], [0, df_pp['oddity_p'].max()], 'k--', alpha=0.3, label='y=x')
axes[1].set_xlabel('Orbital Oddity of p')
axes[1].set_ylabel('Orbital Oddity of p^k')
axes[1].set_title('Prime Power Oddities: oddity(p^k) vs oddity(p)')
axes[1].legend()

plt.tight_layout()
plt.show()

## Near-Trivial: n = 2^a × p

In [ ]:
# For odd primes p < 50 and a = 1..8, examine how class(2^a * p) relates to class(p).

odd_primes = [p for p in range(3, 50) if is_prime(p)]

rows_nt = []
for p in odd_primes:
    class_p = dropping_set(p)
    for a in range(1, 9):
        n = (2 ** a) * p
        rows_nt.append({
            'p': p,
            'a': a,
            'n': n,
            'class_p': class_p,
            'class_n': dropping_set(n),
            'oddity_n': orbital_oddity(n),
        })

df_nt = pd.DataFrame(rows_nt)

# Build pivot table for heatmap: rows = primes p, cols = exponent a, values = class(2^a * p)
pivot_class = df_nt.pivot(index='p', columns='a', values='class_n')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Heatmap of class(2^a * p)
im1 = axes[0].imshow(pivot_class.values, aspect='auto', cmap='viridis')
axes[0].set_xticks(range(len(pivot_class.columns)))
axes[0].set_xticklabels(pivot_class.columns)
axes[0].set_yticks(range(len(pivot_class.index)))
axes[0].set_yticklabels(pivot_class.index)
axes[0].set_xlabel('Exponent a (in 2^a * p)')
axes[0].set_ylabel('Prime p')
axes[0].set_title('Dropping Set of 2^a * p')
plt.colorbar(im1, ax=axes[0], label='Dropping Set')

# Heatmap of oddity(2^a * p)
pivot_odd = df_nt.pivot(index='p', columns='a', values='oddity_n')
im2 = axes[1].imshow(pivot_odd.values, aspect='auto', cmap='plasma')
axes[1].set_xticks(range(len(pivot_odd.columns)))
axes[1].set_xticklabels(pivot_odd.columns)
axes[1].set_yticks(range(len(pivot_odd.index)))
axes[1].set_yticklabels(pivot_odd.index)
axes[1].set_xlabel('Exponent a (in 2^a * p)')
axes[1].set_ylabel('Prime p')
axes[1].set_title('Orbital Oddity of 2^a * p')
plt.colorbar(im2, ax=axes[1], label='Orbital Oddity')

plt.tight_layout()
plt.show()

# Note: For even numbers 2^a * p, the dropping set is always 1 (single division by 2)
# only when a >= 1, so we expect class_n = 1 for all entries.
# The interesting question is what happens after the powers of 2 are divided out.
print("\nClass distribution for near-trivial numbers:")
print(df_nt.groupby('a')['class_n'].value_counts().unstack(fill_value=0))

## Class Multiplication Table?

In [ ]:
# For all pairs of primes (p, q) where p, q < 50,
# build a table of class(p*q) vs (class(p), class(q)).

primes_50 = [p for p in range(3, 50) if is_prime(p)]  # odd primes < 50

rows_mult = []
for p in primes_50:
    for q in primes_50:
        n = p * q
        rows_mult.append({
            'p': p,
            'q': q,
            'n': n,
            'class_p': dropping_set(p),
            'class_q': dropping_set(q),
            'class_pq': dropping_set(n),
            'oddity_pq': orbital_oddity(n),
        })

df_mult = pd.DataFrame(rows_mult)

# Build the "multiplication table": class(p*q) indexed by (class(p), class(q))
# Since class(p) determines the row and class(q) the column, the value is class(p*q).
# If there is a consistent rule, each (row, col) cell should have a single value.

# Check: for each pair (class_p, class_q), what distinct values does class_pq take?
mult_table = df_mult.groupby(['class_p', 'class_q'])['class_pq'].apply(lambda x: sorted(x.unique())).reset_index()
mult_table.columns = ['class_p', 'class_q', 'class_pq_values']
mult_table['num_distinct'] = mult_table['class_pq_values'].apply(len)

print("Class 'Multiplication Table' - distinct class(p*q) values for each (class_p, class_q):")
print(mult_table.to_string(index=False))

print(f"\nTotal (class_p, class_q) pairs: {len(mult_table)}")
print(f"Pairs with unique class_pq: {(mult_table['num_distinct'] == 1).sum()}")
print(f"Pairs with multiple class_pq: {(mult_table['num_distinct'] > 1).sum()}")

# Visualization: Heatmap of the most common class(p*q) for each (class_p, class_q) pair
mode_table = df_mult.groupby(['class_p', 'class_q'])['class_pq'].agg(lambda x: x.mode().iloc[0])
pivot_mult = mode_table.unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(pivot_mult.values, cmap='tab20', aspect='auto')
ax.set_xticks(range(len(pivot_mult.columns)))
ax.set_xticklabels(pivot_mult.columns)
ax.set_yticks(range(len(pivot_mult.index)))
ax.set_yticklabels(pivot_mult.index)
ax.set_xlabel('Dropping Set of q')
ax.set_ylabel('Dropping Set of p')
ax.set_title('Most Common class(p*q) for each (class_p, class_q) pair')
plt.colorbar(im, ax=ax, label='Modal class(p*q)')

# Annotate cells
for i in range(len(pivot_mult.index)):
    for j in range(len(pivot_mult.columns)):
        ax.text(j, i, str(pivot_mult.values[i, j]),
                ha='center', va='center', fontsize=8, color='white',
                fontweight='bold')

plt.tight_layout()
plt.show()

## Statistical Analysis

In [ ]:
# Compute mutual information between factor classes and composite class.
# MI(X; Y) = sum_{x,y} p(x,y) * log(p(x,y) / (p(x)*p(y)))
# We use the semiprime data from above.

def mutual_information(labels_x, labels_y):
    """Compute mutual information between two categorical variables."""
    df_mi = pd.DataFrame({'x': labels_x, 'y': labels_y})
    joint = df_mi.groupby(['x', 'y']).size() / len(df_mi)
    px = df_mi['x'].value_counts(normalize=True)
    py = df_mi['y'].value_counts(normalize=True)

    mi = 0.0
    for (x, y), pxy in joint.items():
        if pxy > 0:
            mi += pxy * np.log2(pxy / (px[x] * py[y]))
    return mi

# MI between class(p) and class(p*q)
mi_p_pq = mutual_information(df_semi['class_p'], df_semi['class_n'])

# MI between class(q) and class(p*q)
mi_q_pq = mutual_information(df_semi['class_q'], df_semi['class_n'])

# MI between (class_p, class_q) combined and class(p*q)
combined_factor_class = list(zip(df_semi['class_p'], df_semi['class_q']))
mi_combined = mutual_information(combined_factor_class, df_semi['class_n'])

# Entropy of class(p*q)
from collections import Counter
counts_pq = Counter(df_semi['class_n'])
total = sum(counts_pq.values())
entropy_pq = -sum((c/total) * np.log2(c/total) for c in counts_pq.values() if c > 0)

print("=== Mutual Information Analysis (semiprimes up to 1000) ===")
print(f"H(class_pq) = {entropy_pq:.4f} bits")
print(f"MI(class_p; class_pq) = {mi_p_pq:.4f} bits")
print(f"MI(class_q; class_pq) = {mi_q_pq:.4f} bits")
print(f"MI((class_p, class_q); class_pq) = {mi_combined:.4f} bits")
print(f"\nNormalized MI (fraction of entropy explained):")
print(f"  By class(p) alone: {mi_p_pq/entropy_pq:.4f}")
print(f"  By class(q) alone: {mi_q_pq/entropy_pq:.4f}")
print(f"  By (class_p, class_q) jointly: {mi_combined/entropy_pq:.4f}")

# Also test using prime power data
mi_pp = mutual_information(df_pp['class_p'], df_pp['class_pk'])
entropy_pk = -sum((c/len(df_pp)) * np.log2(c/len(df_pp))
                   for c in Counter(df_pp['class_pk']).values() if c > 0)
print(f"\n=== Prime Powers ===")
print(f"H(class_pk) = {entropy_pk:.4f} bits")
print(f"MI(class_p; class_pk) = {mi_pp:.4f} bits")
print(f"Normalized: {mi_pp/entropy_pk:.4f}")

## Observations

Key findings from this notebook:

1. **Semiprimes**: Does `class(p*q)` depend predictably on `class(p)` and `class(q)`? The cross-tabulation and mutual information analysis reveal the degree of predictability.

2. **Prime Powers**: How does `class(p^k)` relate to `class(p)` as `k` increases? The scatter plots show whether powers stay in the same class or diverge.

3. **Near-Trivial Composites**: For `n = 2^a * p`, even numbers always have dropping set 1 (single halving step), so the structure is trivial. The interesting patterns emerge when looking at the full Collatz orbit beyond the first drop.

4. **Class Multiplication Table**: If a deterministic rule existed, each `(class_p, class_q)` pair would map to exactly one `class_pq`. The number of pairs with unique vs. multiple outcomes indicates whether such a table is possible.

5. **Mutual Information**: The normalized MI values quantify how much of the composite's classification is explained by knowing the factor classifications.